<a href="https://colab.research.google.com/github/Yah66/VoicePrint-Python/blob/main/index.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import torch
import torchaudio
import torchaudio.transforms as T
from typing import Optional
import matplotlib.pyplot as plt

class ProfessionalAudioProcessor:


    def __init__(
        self,
        sample_rate: int = 16000,
        n_fft: int = 512,
        n_mels: int = 80,
        frame_length_ms: float = 25.0,
        frame_step_ms: float = 10.0,
        window_type: str = 'hamming',
        pre_emphasis: float = 0.0,
        top_db: Optional[float] = None
    ):
        self.sample_rate = sample_rate
        self.n_fft = n_fft
        self.n_mels = n_mels
        self.frame_length_ms = frame_length_ms
        self.frame_step_ms = frame_step_ms
        self.pre_emphasis = pre_emphasis
        self.top_db = top_db

        # number of | in every frame
        self.frame_length = int(frame_length_ms * sample_rate / 1000)  # 400
        self.hop_length = int(frame_step_ms * sample_rate / 1000)      # 160

        # اframe type
        windows = {
            'hamming': torch.hamming_window,
            'hann': torch.hann_window,
            'blackman': torch.blackman_window
        }
        self.window_fn = windows.get(window_type, torch.hamming_window)

        # إنشاء محول الميل مرة واحدة فقط (للكفاءة)
        self.mel_transform = T.MelSpectrogram(
            sample_rate=sample_rate,
            n_fft=n_fft,
            hop_length=self.hop_length,
            n_mels=n_mels,
            window_fn=self.window_fn,
            power=2.0  # (Power Spectrum)
        )

        # to db
        self.to_db = T.AmplitudeToDB(top_db=top_db)

        self._mel_filterbank = None

    def load_audio(self, file_path: str) -> torch.Tensor:
        """upload wav"""
        waveform, sr = torchaudio.load(file_path)

        # Mono conversion
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        # Resampling
        if sr != self.sample_rate:
            waveform = torchaudio.functional.resample(waveform, sr, self.sample_rate)

        # Pre-emphasis (optional)
        if self.pre_emphasis > 0:
            waveform = self._apply_pre_emphasis(waveform)

        return waveform

    def _apply_pre_emphasis(self, waveform: torch.Tensor) -> torch.Tensor:
        """  pre-emphasis y[t] = x[t] - α·x[t-1]"""
        return torch.cat([
            waveform[:, :1],
            waveform[:, 1:] - self.pre_emphasis * waveform[:, :-1]
        ], dim=1)

    def extract_features(self, file_path: str) -> torch.Tensor:
        """
        extract Log-Mel Spectrogram.

        output:
            features: (1, n_mels, num_frames)
        """
        # uplaad
        waveform = self.load_audio(file_path)

        # Mel Spectrogram ( "Framing + Windowing + FFT + Mel" in one step)
        mel_spec = self.mel_transform(waveform)

        # mel to db
        log_mel = self.to_db(mel_spec)

        return log_mel

    def summary(self) -> str:
        """SUMMARY PRINT"""
        duration_per_frame = self.frame_length / self.sample_rate * 1000
        freq_resolution = self.sample_rate / self.n_fft

        return f"""
        ╔══════════════════════════════════════╗
        ║   Audio Processor Configuration      ║
        ╠══════════════════════════════════════╣
        ║ Sample Rate:    {self.sample_rate:>5} Hz           ║
        ║ FFT Points:     {self.n_fft:>5} pts           ║
        ║ Freq Res:       {freq_resolution:>5.2f} Hz/bin     ║
        ║ Frame Length:   {self.frame_length_ms:>5.1f} ms ({self.frame_length} samples) ║
        ║ Frame Step:     {self.frame_step_ms:>5.1f} ms ({self.hop_length} samples) ║
        ║ Mel Filters:    {self.n_mels:>5}              ║
        ║ Window Type:    {self.window_fn.__name__:>12}   ║
        ║ Pre-emphasis:   {self.pre_emphasis:>5}              ║
        ╚══════════════════════════════════════╝
        """

    def visualize(self, file_path: str, save_path: Optional[str] = None):
        """Mel Spectrogram """
        waveform = self.load_audio(file_path)
        features = self.extract_features(file_path)

        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6))

        # signal
        time_axis = torch.arange(waveform.shape[1]) / self.sample_rate
        ax1.plot(time_axis, waveform.squeeze().numpy(), color='#1f77b4', linewidth=0.5)
        ax1.set_title("Raw Waveform - Time Domain", fontsize=14, fontweight='bold')
        ax1.set_xlabel("Time (seconds)")
        ax1.set_ylabel("Amplitude")
        ax1.grid(alpha=0.3)

        # Mel Spectrogram
        im = ax2.imshow(
            features.squeeze().numpy(),
            aspect='auto',
            origin='lower',
            cmap='magma',
            extent=[0, features.shape[2] * self.frame_step_ms / 1000, 0, self.n_mels]
        )
        ax2.set_title("Log-Mel Spectrogram", fontsize=14, fontweight='bold')
        ax2.set_xlabel("Time (seconds)")
        ax2.set_ylabel("Mel Filter Index")
        plt.colorbar(im, ax=ax2, label='dB')

        plt.tight_layout()

        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
            print(f"✅path saved: {save_path}")

        plt.show()


if __name__ == "__main__":
    processor = ProfessionalAudioProcessor()
    print(processor.summary())


        ╔══════════════════════════════════════╗
        ║   Audio Processor Configuration      ║
        ╠══════════════════════════════════════╣
        ║ Sample Rate:    16000 Hz           ║
        ║ FFT Points:       512 pts           ║
        ║ Freq Res:       31.25 Hz/bin     ║
        ║ Frame Length:    25.0 ms (400 samples) ║
        ║ Frame Step:      10.0 ms (160 samples) ║
        ║ Mel Filters:       80              ║
        ║ Window Type:    hamming_window   ║
        ║ Pre-emphasis:     0.0              ║
        ╚══════════════════════════════════════╝
        


In [14]:
!pip install speechbrain

import torch
import torchaudio
from speechbrain.pretrained import EncoderClassifier

# ECAPA-TDNN trained on VoxCeleb
classifier = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir="pretrained_models/ecapa-tdnn"
)


print(f"size {classifier.encode_batch(torch.randn(1,16000)).shape}")

INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Using symlink found at '/content/pretrained_models/ecapa-tdnn/hyperparams.yaml'
INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Using symlink found at '/content/pretrained_models/ecapa-tdnn/embedding_model.ckpt'
INFO:speechbrain.utils.fetching:Fetch mean_var_norm_emb.ckpt: Using symlink found at '/content/pretrained_models/ecapa-tdnn/mean_var_norm_emb.ckpt'
INFO:speechbrain.utils.fetching:Fetch classifier.ckpt: Using symlink found at '/content/pretrained_models/ecapa-tdnn/classifier.ckpt'
INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Using symlink found at '/content/pretrained_models/ecapa-tdnn/label_encoder.ckpt'
INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder


size torch.Size([1, 1, 192])


In [15]:
import torch
import torchaudio

def extract_speaker_embedding(audio_path, classifier, pre_emphasis_coef=0.97):

    waveform, sr = torchaudio.load(audio_path)

    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)

    if sr != 16000:
        waveform = torchaudio.functional.resample(waveform, sr, 16000)


    if torch.max(torch.abs(waveform)) > 0:
        waveform = waveform / torch.max(torch.abs(waveform))


    if pre_emphasis_coef > 0:
        # y[t] = x[t] - α · x[t-1]
        waveform[:, 1:] = waveform[:, 1:] - pre_emphasis_coef * waveform[:, :-1]


    with torch.no_grad():
        embedding = classifier.encode_batch(waveform)

    return embedding.squeeze()

In [16]:
# Extract Speaker Embedding
emb1 = extract_speaker_embedding("/content/record111.wav", classifier)
emb2 = extract_speaker_embedding("/content/record222.wav", classifier)  # same person
emb3 = extract_speaker_embedding("/content/record3.wav", classifier)  # different person

# (Cosine Similarity)
similarity_same = torch.nn.functional.cosine_similarity(emb1.squeeze(), emb2.squeeze(), dim=0)
similarity_diff = torch.nn.functional.cosine_similarity(emb1.squeeze(), emb3.squeeze(), dim=0)

print(f"Same Speaker:     {similarity_same.item():.3f}")   # expected > 0.7
print(f"Diffrent Speakers {similarity_diff.item():.3f}")  # expected < 0.3

Same Speaker:     0.739
Diffrent Speakers 0.134
